In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import re
import pickle
from typing import List, Tuple
from typing import Any
from typing import Dict, Optional, Union
from tqdm import tqdm
from evaluation.clustering import analyze_detection_mask

/home/mballo_sw/Repositories/ecg-seizure-detection/TimeVQVAE-AD/.venv/lib/python3.10/site-packages/x_transformers/x_transformers.py:504: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
/home/mballo_sw/Repositories/ecg-seizure-detection/TimeVQVAE-AD/.venv/lib/python3.10/site-packages/x_transformers/x_transformers.py:528: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
/home/mballo_sw/Repositories/ecg-seizure-detection/TimeVQVAE-AD/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/mballo_sw/Repositories/ecg-seizure-detection/TimeVQVAE-AD/models/stage1/vq.py:231: FutureWarning: `torch.cuda.am

In [ ]:
def load_pkl(fname: str):
    with open(fname, 'rb') as f:
        return pickle.load(f)

def is_windowed(filename: Union[str, Path]) -> bool:
    """Check if file is windowed based on filename.
    Returns True if windowed, False if no_window."""
    filename_str = str(filename)
    return 'no_window' not in filename_str

def get_events(mask) -> List[Tuple[int, int]]:
    # Convert tensor to numpy array if needed
    if hasattr(mask, 'cpu'):  # PyTorch tensor
        mask = mask.cpu().numpy()
    elif hasattr(mask, 'numpy'):  # TensorFlow tensor
        mask = mask.numpy()

    # Ensure it's a 1D array
    mask = np.asarray(mask).flatten()

    events = []
    in_event = False
    for i, v in enumerate(mask):
        if v and not in_event:
            start = i
            in_event = True
        elif not v and in_event:
            end = i - 1
            events.append((start, end))
            in_event = False
    if in_event:
        events.append((start, len(mask) - 1))
    return events

def predict_anomalies(a_final, threshold):
    # Convert tensor to numpy if needed
    if hasattr(a_final, 'cpu'):
        a_final = a_final.cpu().numpy()
    elif hasattr(a_final, 'numpy'):
        a_final = a_final.numpy()
    else:
        a_final = np.asarray(a_final)
    
    return a_final > threshold


def evaluate_event_detection(pred: np.ndarray, truth: np.ndarray, fs: Optional[float] = None, 
                            is_clustered: bool = False) -> Dict[str, object]:
    """
    Evaluate event-based detection metrics.
    
    Args:
        pred: Predicted anomaly mask
        truth: Ground truth anomaly mask
        fs: Sampling frequency (Hz)
        is_clustered: If True, append '_cluster' suffix to metric names
    
    Returns:
        Dictionary with detection metrics
    """
    truth_events = get_events(truth)
    pred_events = get_events(pred)
    TP = sum(1 for (ts, te) in truth_events if pred[ts:te+1].any())
    FN = len(truth_events) - TP
    FP = sum(1 for (ps, pe) in pred_events if not truth[ps:pe+1].any())
    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    
    # Add suffix if clustered
    suffix = '_cluster' if is_clustered else ''
    
    metrics: Dict[str, object] = {
        f'truth_events{suffix}': truth_events,
        f'pred_events{suffix}': pred_events,
        f'TP_events{suffix}': TP, 
        f'FN_events{suffix}': FN, 
        f'FP_events{suffix}': FP,
        f'sensitivity{suffix}': sensitivity,
    }
    if fs is not None:
        total_hours = len(truth) / fs / 3600.0
        metrics[f'false_alarm_rate_per_hour{suffix}'] = (FP / total_hours) if total_hours > 0 else float('inf')
    return metrics



def get_clustered_mask_from_representatives(
    representatives: List[Dict[str, Any]], 
    total_samples: int,
    fs: float
) -> List[bool]:
    """
    Convert cluster representatives back to a boolean mask.
    Each representative's cluster span is marked as True.
    
    Args:
        representatives: List of representative dicts from clustering analysis
        total_samples: Total length of the original mask
        fs: Sampling rate in Hz
    
    Returns:
        Boolean mask of length total_samples with clustered anomalies marked as True
    """
    mask = [False] * total_samples
    
    for rep in representatives:
        # Get cluster time span
        start_sec = rep.get('cluster_start_seconds', rep['location_time_seconds'])
        end_sec = rep.get('cluster_end_seconds', rep['location_time_seconds'])
        
        # Convert to sample indices
        start_idx = int(start_sec * fs)
        end_idx = int(end_sec * fs)
        
        # Clamp to valid range
        start_idx = max(0, min(start_idx, total_samples - 1))
        end_idx = max(0, min(end_idx, total_samples - 1))
        
        # Mark cluster span as True
        for i in range(start_idx, end_idx + 1):
            mask[i] = True
    
    return mask


def analyze_detection_mask_return_mask(
    mask,
    fs: float,
    file_id: str,
    scores: Optional[List[float]],
    subject_id: str,
    gt_intervals: Optional[List[Tuple[float, float]]] = None,
    fixed_strategy: Optional[str] = None,
    true_anomaly_samples: Optional[int] = None,
    true_anomaly_events: Optional[int] = None
) -> Tuple[List[bool], Dict[str, Any]]:
    """
    Same as analyze_detection_mask but returns (clustered_mask, full_results).
    
    Returns:
        Tuple of (boolean mask with clustered anomalies, full analysis results dict)
    """
    # Run the full analysis
    results = analyze_detection_mask(
        mask=mask,
        fs=fs,
        file_id=file_id,
        scores=scores,
        subject_id=subject_id,
        gt_intervals=gt_intervals,
        output_folder=None,  # Don't save files
        fixed_strategy=fixed_strategy,
        true_anomaly_samples=true_anomaly_samples,
        true_anomaly_events=true_anomaly_events
    )
    
    # Extract representatives
    representatives = results['best_results']['representatives']
    
    # Convert to mask
    clustered_mask = get_clustered_mask_from_representatives(
        representatives, 
        len(mask), 
        fs
    )
    
    return clustered_mask, results

In [3]:
# Get all the important files
runs_dir = Path('evaluation/results/runs')

# Get files from both subdirectories
no_window_dir = runs_dir / 'no_window_joint_anomaly'
window_dir = runs_dir / 'window_joint_anomaly'

# Get all pickle files from both directories
no_window_files = list(no_window_dir.glob('*.pkl')) if no_window_dir.exists() else []
window_files = list(window_dir.glob('*.pkl')) if window_dir.exists() else []

# Filter for joint_anomaly_score files only (exclude predicted_seizures files)
pattern = re.compile(r'.*-joint_anomaly_score\.pkl$')
joint_anomaly_files = [f for f in no_window_files + window_files if pattern.match(f.name)]

# Exclude specific runs (note: files use format like "122_run-29" not "sub-122_run-29")
exclude_pattern = re.compile(r'^(099_run-01|114_run-03|115_run-11|115_run-32|117_run-13|118_run-07|119_run-24|119_run-36|123_run-22|124_run-19|124_run-43|124_run-63|125_run-36|125_run-67)_')
joint_anomaly_files = [f for f in joint_anomaly_files if not exclude_pattern.match(f.name)]

len(joint_anomaly_files)

1478

In [16]:
configs = [(0.75, "time_120s"), (0.7, "time_60s"), (0.65, "time_30s")]
results = []

# Calculate total iterations for progress bar
total_iterations = sum(1 for scale, clustering_strategy in configs 
                      for file in joint_anomaly_files if is_windowed(file))

with tqdm(total=total_iterations, desc="Processing files") as pbar:
    for scale, clustering_strategy in configs:
        for file in joint_anomaly_files:
            if not is_windowed(file):
                continue
            file_data = load_pkl(file)
            predicted = predict_anomalies(file_data.get("a_final"), file_data.get("final_threshold")*scale)
            
            # Evaluate detection metrics for non-clustered (baseline)
            result = evaluate_event_detection(predicted, file_data.get("Y"), fs=8, is_clustered=False)
            result["scale"] = scale
            result["subject_id"] = file_data.get("subject_id")
            result["run_id"] = file_data.get("run_id")
            result["window"] = is_windowed(file)
            result["duration_seconds"] = len(file_data.get("Y")) / 8.0
            result["predicted_anomalies"] = predicted
            result["real_anomalies"] = file_data.get("Y")

            # Apply clustering
            clustered_mask, ergebnis = analyze_detection_mask_return_mask(
                mask=predicted,
                fs=8.0,
                fixed_strategy=clustering_strategy,
                scores=file_data.get("a_final"),
                file_id=result["run_id"],
                subject_id=result["subject_id"],
            )
            # Convert clustered_mask to numpy array
            clustered_mask = np.array(clustered_mask)
            
            # Evaluate detection metrics for clustered results
            clustered_metrics = evaluate_event_detection(clustered_mask, file_data.get("Y"), fs=8, is_clustered=True)
            result.update(clustered_metrics)
            
            result["predicted_clusters"] = clustered_mask
            results.append(result)
            
            # Update progress bar with file info
            pbar.set_postfix({'file': file.stem, 'scale': scale})
            pbar.update(1)

Processing files: 100%|██████████| 2217/2217 [02:22<00:00, 15.51it/s, file=106_run-08_window-joint_anomaly_score, scale=0.65] 


In [ ]:
len(results[0].get("predicted_clusters")), len(results[0].get("predicted_clusters"))

167424

In [17]:
results_df = pd.DataFrame(results)
results_df.head()


,truth_events,pred_events,TP_events,FN_events,FP_events,sensitivity,false_alarm_rate_per_hour,scale,subject_id,run_id,...,predicted_anomalies,real_anomalies,truth_events_cluster,pred_events_cluster,TP_events_cluster,FN_events_cluster,FP_events_cluster,sensitivity_cluster,false_alarm_rate_per_hour_cluster,predicted_clusters
0,[],"[(4160, 4223), (7808, 7871), (18176, 18239), (...",0,0,56,0.0,9.633028,0.75,sub-117,run-17,...,"[False, False, False, False, False, False, Fal...","[tensor(0), tensor(0), tensor(0), tensor(0), t...",[],"[(4160, 4224), (7808, 7872), (18176, 18240), (...",0,0,28,0.0,4.816514,"[False, False, False, False, False, False, Fal..."
1,"[(147568, 152743)]","[(3968, 4031), (6848, 6911), (8000, 8063), (97...",0,1,48,0.0,8.247423,0.75,sub-115,run-09,...,"[False, False, False, False, False, False, Fal...","[tensor(0), tensor(0), tensor(0), tensor(0), t...","[(147568, 152743)]","[(3968, 4032), (6848, 6912), (8000, 8064), (97...",0,1,29,0.0,4.982818,"[False, False, False, False, False, False, Fal..."
2,[],"[(5504, 5567), (6144, 6207), (7488, 7551), (16...",0,0,6,0.0,6.081081,0.75,sub-121,run-16,...,"[False, False, False, False, False, False, Fal...","[tensor(0), tensor(0), tensor(0), tensor(0), t...",[],"[(5504, 6208), (7488, 7552), (16640, 17088), (...",0,0,4,0.0,4.054054,"[False, False, False, False, False, False, Fal..."
3,[],"[(1024, 1151), (2048, 2111), (2309, 2367), (28...",0,0,13,0.0,13.175676,0.75,sub-125,run-06,...,"[False, False, False, False, False, False, Fal...","[tensor(0), tensor(0), tensor(0), tensor(0), t...",[],"[(1024, 1152), (2048, 2880), (8192, 8256), (10...",0,0,7,0.0,7.094595,"[False, False, False, False, False, False, Fal..."
4,[],"[(1728, 1855), (1984, 2047), (2112, 2139)]",0,0,3,0.0,30.000000,0.75,sub-125,run-85,...,"[False, False, False, False, False, False, Fal...","[tensor(0), tensor(0), tensor(0), tensor(0), t...",[],"[(1728, 2140)]",0,0,1,0.0,10.000000,"[False, False, False, False, False, False, Fal..."


In [22]:
len(results_df["subject_id"].unique())

28

In [ ]:
sum(results_df["duration_seconds"])

29093976.0

739

In [41]:
mean_run_length = sum(results_df["duration_seconds"]) / len(results_df[["subject_id", "run_id"]].drop_duplicates())
mean_sub_length = sum(results_df["duration_seconds"]) / len(results_df["subject_id"].unique())
ratio = mean_sub_length / mean_run_length
eval = ratio * np.array([0.005, 0.01, 0.03])
eval

array([0.13196429, 0.26392857, 0.79178571])

In [43]:
(mean_run_length * eval[0])/60

86.58921428571428

In [44]:
(mean_sub_length*0.005)/60

86.58921428571429

In [6]:
results_df.describe()

,TP_events,FN_events,FP_events,sensitivity,false_alarm_rate_per_hour,scale,duration_seconds,TP_events_cluster,FN_events_cluster,FP_events_cluster,sensitivity_cluster,false_alarm_rate_per_hour_cluster
count,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000
mean,0.246667,0.023333,176.413333,0.126333,49.830712,0.700000,13153.680000,0.246667,0.023333,74.090000,0.126333,20.163335
std,1.043769,0.171913,324.232991,0.329802,34.918576,0.040893,19133.180158,1.043769,0.171913,145.080517,0.329802,14.760553
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.650000,1296.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,25.000000,0.000000,11.205530,0.650000,3504.000000,0.000000,0.000000,10.000000,0.000000,6.590149
50%,0.000000,0.000000,69.000000,0.000000,50.000000,0.700000,3552.000000,0.000000,0.000000,26.000000,0.000000,16.324777
75%,0.000000,0.000000,108.250000,0.000000,83.060764,0.750000,19080.000000,0.000000,0.000000,53.500000,0.000000,37.179591
max,10.000000,2.000000,1914.000000,1.000000,111.986301,0.750000,80952.000000,10.000000,2.000000,955.000000,1.000000,50.000000


In [7]:
results_df.columns

Index(['truth_events', 'pred_events', 'TP_events', 'FN_events', 'FP_events',
       'sensitivity', 'false_alarm_rate_per_hour', 'scale', 'subject_id',
       'run_id', 'window', 'duration_seconds', 'predicted_anomalies',
       'real_anomalies', 'truth_events_cluster', 'pred_events_cluster',
       'TP_events_cluster', 'FN_events_cluster', 'FP_events_cluster',
       'sensitivity_cluster', 'false_alarm_rate_per_hour_cluster',
       'predicted_clusters'],
      dtype='object')

In [8]:
results[0].keys()

dict_keys(['truth_events', 'pred_events', 'TP_events', 'FN_events', 'FP_events', 'sensitivity', 'false_alarm_rate_per_hour', 'scale', 'subject_id', 'run_id', 'window', 'duration_seconds', 'predicted_anomalies', 'real_anomalies', 'truth_events_cluster', 'pred_events_cluster', 'TP_events_cluster', 'FN_events_cluster', 'FP_events_cluster', 'sensitivity_cluster', 'false_alarm_rate_per_hour_cluster', 'predicted_clusters'])

In [9]:
results[0].get('truth_events')

[]

In [10]:
len(results[0].get('pred_events'))

27

In [11]:
len(results[0].get('pred_events_cluster'))

16

In [12]:
results_df_part = results_df[(results_df["scale"] == 0.7) &results_df["window"]]
results_df_part.describe()

,TP_events,FN_events,FP_events,sensitivity,false_alarm_rate_per_hour,scale,duration_seconds,TP_events_cluster,FN_events_cluster,FP_events_cluster,sensitivity_cluster,false_alarm_rate_per_hour_cluster
count,100.000000,100.0,100.000000,100.000000,100.000000,1.000000e+02,100.00000,100.000000,100.0,100.000000,100.000000,100.000000
mean,0.270000,0.0,188.530000,0.140000,54.264559,7.000000e-01,13153.68000,0.270000,0.0,57.710000,0.140000,15.839859
std,1.090408,0.0,286.320843,0.348735,18.860370,2.231632e-16,19197.49355,1.090408,0.0,86.350039,0.348735,3.654369
min,0.000000,0.0,4.000000,0.000000,9.375000,7.000000e-01,1296.00000,0.000000,0.0,0.000000,0.000000,0.000000
25%,0.000000,0.0,44.750000,0.000000,38.579022,7.000000e-01,3504.00000,0.000000,0.0,14.000000,0.000000,13.718369
50%,0.000000,0.0,68.000000,0.000000,52.529989,7.000000e-01,3552.00000,0.000000,0.0,18.000000,0.000000,16.324777
75%,0.000000,0.0,214.250000,0.000000,66.277296,7.000000e-01,19080.00000,0.000000,0.0,74.000000,0.000000,18.212024
max,10.000000,0.0,1534.000000,1.000000,99.193548,7.000000e-01,80952.00000,10.000000,0.0,408.000000,1.000000,24.193548


In [13]:
np.mean(results_df_part["sensitivity_cluster"])

0.14

In [14]:
results_df_part.columns

Index(['truth_events', 'pred_events', 'TP_events', 'FN_events', 'FP_events',
       'sensitivity', 'false_alarm_rate_per_hour', 'scale', 'subject_id',
       'run_id', 'window', 'duration_seconds', 'predicted_anomalies',
       'real_anomalies', 'truth_events_cluster', 'pred_events_cluster',
       'TP_events_cluster', 'FN_events_cluster', 'FP_events_cluster',
       'sensitivity_cluster', 'false_alarm_rate_per_hour_cluster',
       'predicted_clusters'],
      dtype='object')

In [15]:
type(results_df_part["truth_events"][0])

KeyError: 0

In [ ]:
results_df_part["has_anomalie"] = results_df_part["truth_events"].apply(lambda x: len(x) > 0)

/tmp/ipykernel_566812/3068043680.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results_df_part["has_anomalie"] = results_df_part["truth_events"].apply(lambda x: len(x) > 0)


In [ ]:
np.sum(results_df_part["has_anomalie"])

81

In [ ]:
np.sum(results_df_part["truth_events"].apply(len))

148